# Text Generation with SNEPPX-Algo

Greedy, sampling (top-k/top-p), beam search, and live token streaming using
`SneppX_ALG.interface_bindings.generation`. The NumPy sampling loop runs
without the C backend.

In [ ]:
import numpy as np
from SneppX_ALG import Transformer
from SneppX_ALG.interface_bindings.generation import (
    generate, GenerationConfig, TextStreamer, TokenStreamer,
)

model = Transformer(vocab_size=200, dim=64, num_heads=4, num_layers=2,
                    ffn_dim=128, max_seq_len=64, dropout=0.0)
model.eval()

## Greedy search

In [ ]:
cfg = GenerationConfig(max_new_tokens=24, do_sample=False)
ids = [1, 2, 3]
out = generate(model, ids, generation_config=cfg)
print('output_ids shape:', out['output_ids'].shape)

## Sampling: top-k + top-p (nucleus)

In [ ]:
cfg = GenerationConfig(
    max_new_tokens=48, do_sample=True, temperature=0.8,
    top_k=40, top_p=0.9, repetition_penalty=1.15,
)
out = generate(model, ids, generation_config=cfg)
print('final length:', out['output_ids'].shape)

## Beam search

In [ ]:
cfg = GenerationConfig(
    max_new_tokens=32, num_beams=3, length_penalty=0.7, early_stopping=True,
)
out = generate(model, ids, generation_config=cfg)
print('beam result:', out['output_ids'].shape)

## Stream tokens live (CPU-safe, pure NumPy)

In [ ]:
from SneppX_ALG.interface_bindings.tokenizer import SimpleTokenizer
tok = SimpleTokenizer(vocab_size=200)
stream = TextStreamer(tokenizer=tok, skip_prompt=True)
cfg = GenerationConfig(max_new_tokens=20)
generate(model, ids, generation_config=cfg, streamer=stream)
# tokens are printed as they are produced

## Batch generation (variable-length prompts)

In [ ]:
from SneppX_ALG.interface_bindings.generation import batch_generate
prompts = [[1, 2, 3], [4, 5, 6, 7, 8], [9, 10]]
cfg = GenerationConfig(max_new_tokens=16, temperature=0.7)
out = batch_generate(model, prompts, generation_config=cfg)
print('batched shape:', out['output_ids'].shape)